# SZA illumination-leak diagnostic

**Why this exists.** In the all-patches matched-pairs analysis the **radiance** drift spectrum shows a positive bias (most of its uncertainty above 0), while **reflectance** keeps the slight negative bias seen in the one-patch analysis. A true *sensor* radiometric drift would appear in **both** products (L2A reflectance is derived from L1B radiance). Appearing in radiance **only** points to a term that reflectance normalises away — dominantly the **solar illumination** (radiance scales with cos(SZA)).

Matched pairs bound `|ΔSZA| <= sza_tol`, but that only bounds the **magnitude** — the **signed** residual `SZA(t2) − SZA(t1)` can still average away from zero. If the later scene tends to have *higher sun* (lower SZA) than the earlier one, and that tendency grows with Δt, radiance ΔR picks up a spurious positive slope that reflectance never sees.

This notebook (1) measures the signed ΔSZA and its correlation with Δt, (2) shows its distribution, and (3) recomputes the radiance drift spectrum with a first-order **cos(SZA) illumination correction**, next to the reflectance reference.

> Runs off `data/delta_r/aggregated/*_delta.csv` — set the patch size in `run_config.yaml` and complete the analyze stage first.

In [ ]:
# Repo-relative bootstrap: find the repo root (holds run_config.yaml) and import.
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / "run_config.yaml").exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, theilslopes

from emit_pics import paths
from emit_pics.run_config import load_config
from emit_pics.matched_pairs_drift_analysis import (
    load_aggregated_delta_tables, compute_drift_spectrum, DAYS_PER_YEAR)

cfg   = load_config()
ALPHA = cfg.alpha
SIZE  = cfg.patch_size

# Pool the aggregated (one-row-per-pair) delta tables across all PICS of this size.
df = load_aggregated_delta_tables(paths.DELTA_AGG_DIR, SIZE)
if df.empty:
    raise SystemExit(
        f"No aggregated delta tables for size {SIZE!r} in {paths.DELTA_AGG_DIR}.\n"
        "Run the analyze stage (scripts/run_PICS_allpatches_aggregate_pairs.py) first.")

n_pairs = df.groupby(["PICS_ID", "pair_id"]).ngroups
print(f"patch size {SIZE} km | {len(df)} rows | {df['PICS_ID'].nunique()} PICS | "
      f"{n_pairs} unique pairs | {df['band_index'].nunique()} channels")

In [ ]:
# --- Adjustable plot y-limits -------------------------------------------------
# The drift-spectrum panels otherwise autoscale to the most extreme (often
# numerically unstable) channel, which flattens the ~0.5 %/yr differences between
# the series. Set tight limits here; use None to autoscale. Re-run the plot cells
# after changing these.
SPECTRUM_YLIM = (-1.5, 1.5)   # drift-spectrum panels, %/yr
DIAG_YLIM     = None          # signed-ΔSZA / flux-ratio scatter (degrees or %)

## 1. Signed ΔSZA per pair
`SZA_t1`/`SZA_t2` are per-scene mean |SZA| (t1 = earlier scene). `dSZA_signed > 0` means the **later** scene had a **higher** SZA (lower sun → less radiance → a *negative* radiance-ΔR push); `dSZA_signed < 0` is the opposite and is what would create the observed **positive** radiance bias.

In [ ]:
# One row per unique acquisition pair (SZA/Δt are identical across a pair's bands).
pairs = df.drop_duplicates(subset=["PICS_ID", "pair_id"]).copy()
pairs["dSZA_signed"] = pairs["SZA_t2"] - pairs["SZA_t1"]
pairs["dcos_signed"] = (np.cos(np.radians(pairs["SZA_t2"]))
                        - np.cos(np.radians(pairs["SZA_t1"])))

print(f"{len(pairs)} unique pairs")
print(f"signed ΔSZA :  mean {pairs['dSZA_signed'].mean():+.3f}°   "
      f"median {pairs['dSZA_signed'].median():+.3f}°")
print(f"|ΔSZA| max  :  {pairs['dSZA_signed'].abs().max():.2f}°   "
      f"(should be <= the sza_tol used when matching)")

## 2. Does signed ΔSZA correlate with Δt?
A non-zero Theil–Sen slope / significant Kendall τ here is the smoking gun: it means the illumination mismatch is **systematically tied to the time baseline**, so it maps straight into an apparent radiance drift.

In [ ]:
m = pairs["dSZA_signed"].notna() & pairs["dt_days"].notna()
x = pairs.loc[m, "dt_days"].to_numpy()
y = pairs.loc[m, "dSZA_signed"].to_numpy()
tau, p = kendalltau(x, y)
slope, intc, lo, hi = theilslopes(y, x)
print(f"signed ΔSZA vs Δt :  Theil–Sen slope {slope * DAYS_PER_YEAR:+.4f} °/yr   "
      f"Kendall τ = {tau:+.3f}   p = {p:.2g}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(x, y, s=12, alpha=0.5)
xr = np.array([x.min(), x.max()])
ax[0].plot(xr, intc + slope * xr, "k-", lw=1.5)
ax[0].axhline(0.0, color="0.6", lw=0.8)
ax[0].set_xlabel("Δt (days)")
ax[0].set_ylabel("signed ΔSZA = SZA(t2) − SZA(t1)  [°]")
ax[0].set_title(f"signed ΔSZA vs Δt   (τ={tau:+.2f}, p={p:.2g})")
if DIAG_YLIM is not None:
    ax[0].set_ylim(DIAG_YLIM)
ax[1].hist(y, bins=40, color="tab:blue", alpha=0.85)
ax[1].axvline(0.0, color="k", lw=1.0)
ax[1].axvline(float(np.mean(y)), color="tab:red", lw=1.5, ls="--",
             label=f"mean {np.mean(y):+.2f}°")
ax[1].set_xlabel("signed ΔSZA [°]")
ax[1].set_ylabel("pairs")
ax[1].set_title("distribution of signed ΔSZA")
ax[1].legend()
plt.tight_layout()
plt.show()

## 3. cos(SZA) illumination correction
First order, at-sensor radiance ∝ cos(SZA). We divide the illumination ratio `cos(SZA_t2)/cos(SZA_t1)` out of each pair's radiance % change and refit the drift spectrum. If the positive radiance bias is an SZA leak, the corrected radiance spectrum should collapse toward the reflectance reference.

In [ ]:
def remove_illumination(delta_pct, ratio):
    # Undo a multiplicative illumination factor  ratio = illum(t2) / illum(t1)
    # from a percent change. Measured (1 + dR/100) = ratio * (true ratio), so the
    # illumination-free percent change is 100 * ((1 + dR/100) / ratio - 1).
    delta_pct = np.asarray(delta_pct, dtype=float)
    ratio     = np.asarray(ratio, dtype=float)
    return 100.0 * ((1.0 + delta_pct / 100.0) / ratio - 1.0)


def median_drift(spec):
    return float(np.nanmedian(spec["slope_pct_yr"]))


def plot_spectra(specs_labels_colors, title, ylim=None):
    # specs_labels_colors: list of (spectrum_df, label, color); ylim: (lo, hi) or None
    fig, ax = plt.subplots(figsize=(12, 5))
    for spec, lab, c in specs_labels_colors:
        s = spec.dropna(subset=["slope_pct_yr"]).sort_values("wavelength")
        ax.plot(s["wavelength"], s["slope_pct_yr"], ".", ms=5, color=c, label=lab)
    ax.axhline(0.0, color="k", lw=0.8)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("drift (%/yr)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# cos(SZA) illumination ratio per row (identical across a pair's bands).
f_sza = (np.cos(np.radians(df["SZA_t2"])) / np.cos(np.radians(df["SZA_t1"]))).to_numpy()

df_corr = df.copy()
df_corr["delta_rad_pct_median"] = remove_illumination(df["delta_rad_pct_median"], f_sza)

spec_rad      = compute_drift_spectrum(df,      "rad", alpha=ALPHA)
spec_rad_corr = compute_drift_spectrum(df_corr, "rad", alpha=ALPHA)
spec_rfl      = compute_drift_spectrum(df,      "rfl", alpha=ALPHA)

print(f"median radiance drift  original ............ {median_drift(spec_rad):+.3f} %/yr")
print(f"median radiance drift  cos(SZA)-corrected .. {median_drift(spec_rad_corr):+.3f} %/yr")
print(f"median reflectance drift (reference) ....... {median_drift(spec_rfl):+.3f} %/yr")
n_pos = int((spec_rad["slope_pct_yr"] > 0).sum())
n_pos_c = int((spec_rad_corr["slope_pct_yr"] > 0).sum())
n_fit = int(spec_rad["slope_pct_yr"].notna().sum())
print(f"channels with positive radiance drift: {n_pos}/{n_fit} -> {n_pos_c}/{n_fit} after correction")

In [ ]:
plot_spectra(
    [(spec_rad,      "radiance — original",           "tab:red"),
     (spec_rad_corr, "radiance — cos(SZA)-corrected", "tab:blue"),
     (spec_rfl,      "reflectance (reference)",       "tab:green")],
    "Radiance drift spectrum: raw vs cos(SZA) illumination-corrected",
    ylim=SPECTRUM_YLIM)

## How to read this
- **Signed ΔSZA has mean ≈ 0 and no Δt correlation** → SZA is *not* the leak; look at Earth–Sun distance (see `earth_sun_distance_diagnostic.ipynb`) or a genuine radiance effect.
- **Signed ΔSZA trends with Δt and the corrected radiance spectrum drops onto the reflectance curve** → the positive radiance bias was an illumination artifact, not sensor drift. Tightening `matching.sza_tol` (e.g. 2° → 1°) should shrink it directly, and the pooled **BRDF** model — which fits illumination explicitly — is the principled fix.
- This correction is first-order (Lambertian cos(SZA)); treat it as a diagnostic, not a final calibrated product.